In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-ad-starting-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_dirname_output = './output'
    str_model = '01_ad'
    # IDs
    str_id = 'uniqueid'
    str_datecol = 'applicationdate__app'
    str_target = 'target'
    
    # rm
    print('Get columns to ignore...')
    # commented columns in this list means they are considered for feature selection
    list_cols_ignore = [
        # IDs
        str_id,
        str_target,
        str_datecol,
        'year_month',
        'data_set',
        'year',
        'factor',
        # nan in production
        'linkf012__tu',
        'linkf027__tu',
        'linkf029__tu',
        'linkd001__tu',
        # we already have a bitdebtor__app
        'bitdebtor__tu',
        'bitdebtor__base',
        # drifting
#         'addrinputavalue__ln',
#         'addrcurrentavmvalue12month__ln',
#         'addrinputblockratio__ln',
#         'addrinputcountyratio__ln',
#         'addrinputtractratio__ln',
#         'bc101s__tu',
#         'co05s__tu',
#         'bc103s__tu',
#         'rev319__tu',
#         'revs904__tu',
#         'agg908__tu',
#         'rev319__tu',
#         'rev328__tu',
#         'agg911__tu',
#         'agg906__tu',
#         'rev318__tu',
#         'jt20s__tu',
#         'agg907__tu',
        # structure features
#         'strname__app',
#         'bitdealertrack__app',
#         'bitdealerapplicantsamezip__app',
#         'bitdealerapplicantsamecity__app',
#         'bitdealerapplicantsamestate__app',
#         'dealerstate__app',
#         'strdealershiptrackertype__app',
#         'bitdebtor__app',
#         'ENG-applicationdate__app_month',
#         'ENG-applicationdate__app_quarter',
        'inttype__app',
        'fltapprovedloantovalue__app',
        'fltapprovedservicecontract__app',
        'fltservicecontract__app',
        'fltinsuredlifeamount__app',
        'fltgapinsurance__app',
        'fltapprovedpricewholesale__app',
        'fltapproveddowntotal__app',
        'fltapprovedpayment__app',
        'fltamountfinanced__app',
        'intservicecontractmileageaddon__app',
        'bigdealertypeid__app',
        'bitmaintenanceagreement__app',
        'bitservicecontract__app',
        'bitrouteone__app',
        'intterm__app',
        'fltinsureddisabilityamount__app',
        'fltdowncash__app',
        'fltallowance__app',
        'applicationdate__app',
        'dealerstampcreation__app'
        # engineered
        'ENG-payment_to_income',
        'ENG-loan_to_value',
        'ENG-vehicle_age',
        'ENG-dealership_age',
        # income
#         'fltgrossmonthly__income_sum',
#         'fltgrossmonthly__income_count',
        # linkc
        'linkchit__tu',
        'linkc051__tu',
        'linkc007__tu',
        'linkc004__tu',
        'linkc048__tu',
        'linkc039__tu',
        'linkc038__tu',
        'linkc020__tu',
        'linkc005__tu',
        'linkc030__tu',
        'linkc035__tu',
        'linkc034__tu',
        'linkc033__tu',
        'linkc032__tu',
        'linkc031__tu',
        'linkc024__tu',
        'linkc029__tu',
        'linkc028__tu',
        'linkc027__tu',
        'linkc026__tu',
        'linkc040__tu',
        'linkc041__tu',
        'linkc042__tu',
        'linkc043__tu',
        'linkc049__tu',
        'linkc047__tu',
        'linkc046__tu',
        'linkc045__tu',
        'linkc044__tu',
        'linkc006__tu',
        'linkc010__tu',
        'linkc037__tu',
        'linkc011__tu',
        'linkc012__tu',
        'linkc050__tu',
        'linkc009__tu',
        'linkc008__tu',
        'linkc036__tu',
        'linkc025__tu',
        # senior management
        'bitdealerapplicantsamestate__app',
        'bitdealerapplicantsamecity__app',
        'fltapprovedservicecontract__app',
        'fltinsuredlifeamount__app',
        'fltservicecontract__app',
        'bitrouteone__app',
        'bitservicecontract__app',
        'amtfinanced__app',
        'bitmaintenanceagreement__app',
        'fltapprovedloantovalue__app',
        'ENG-payment_to_income',
        'fltinsureddisabilityamount__app',
        'bigdealertypeid__app',
        'fltapproveddowntotal__app',
        'fltdocumentfee__app',
        'payment__app',
        'fltlicensefee__app',
        'intservicecontractterm__app',
        'fltgapinsurance__app',
        'inttermbump__app',
        'fltallowance__app',
        'dti__app',
        'intterm__app',
        'payment__app',
        'intservicecontractmileageaddon__app',
        'intservicecontractterm__app',
        'intservicecontractmileage__app',
        'fltgapinsurance__app',
        'inttermbump__app',
        'bitnew__app',
        # 80% or more NaN in production - LN
        'bankcard_reason3__ln',
        'telecommunications_score__ln',
        'short_term_lending_reason5__ln',
        'short_term_lending_reason4__ln',
        'short_term_lending_reason3__ln',
        'short_term_lending_reason2__ln',
        'short_term_lending_reason1__ln',
        'short_term_lending_score__ln',
        'bankcard_reason5__ln',
        'bankcard_reason4__ln',
        'bankcard_reason2__ln',
        'telecommunications_reason2__ln',
        'bankcard_reason1__ln',
        'bankcard_score__ln',
        'auto_reason5__ln',
        'auto_reason4__ln',
        'auto_reason2__ln',
        'auto_reason1__ln',
        'auto_score__ln',
        'analyticsmatchkey__ln',
        'intscore__ln', # score
        'telecommunications_reason1__ln',
        'auto_reason3__ln',
        'telecommunications_reason3__ln',
        'alert2__ln',
        'telecommunications_reason4__ln',
        'alert10__ln',
        'alert9__ln',
        'alert8__ln',
        'alert6__ln',
        'attribute_index__ln',
        'alert5__ln',
        'alert4__ln',
        'alert3__ln',
        'alert7__ln',
        'alert1__ln',
        'crossindustry_reason1__ln',
        'crossindustry_reason5__ln',
        'crossindustry_index__ln',
        'crossindustry_score__ln',
        'crossindustry_score_name__ln',
        'crossindustry_reason2__ln',
        'crossindustry_reason3__ln',
        'crossindustry_reason4__ln',
        'telecommunications_reason5__ln'
        # 80% or more NaN in production - TU
        'au921b__tu',
        'p03h__tu',
        'linkahit__tu',
        'linkbhit__tu',
        'iscrhit__tu',
        'bctrd__tu',
        'insttrd__tu',
        'nomttrd__tu',
        'revtrd__tu',
        'rtltrd__tu',
        'bcpmtstr__tu',
        'bcpmtnum__tu',
        'permid__tu',
        'partyid__tu',
        'acctid__tu',
        'tuseqeuncenumber__tu',
        'scr_linka2__tu',
        'scr_alta2__tu',
        'scr_linkb2__tu',
        'scr_altb2__tu',
        'cvlst_s1__tu',
        'cvlst_i1__tu',
        'eadms1__tu',
        'p03g__tu',
        'p03f__tu',
        'p03e__tu',
        'ad14h__tu',
        'ad13f__tu',
        'ad13g__tu',
        'ad13h__tu',
        'ad14a__tu',
        'ad14b__tu',
        'ad14c__tu',
        'ad14d__tu',
        'ad14e__tu',
        'ad14f__tu',
        'ad14g__tu',
        'p02a__tu',
        'p03d__tu',
        'p02b__tu',
        'p02c__tu',
        'p02d__tu',
        'p02e__tu',
        'p02f__tu',
        'p02g__tu',
        'p02h__tu',
        'p03a__tu',
        'p03b__tu',
        'p03c__tu',
        'zip_code__tu',
        'cvau__tu',
        'cvam__tu',
        'bc999s__tu',
        'at921c__tu',
        'at933s__tu',
        'at933b__tu',
        'at933c__tu',
        'au900s__tu',
        'au900b__tu',
        'au900c__tu',
        'bc900s__tu',
        'bc900b__tu',
        'bc900c__tu',
        'rl900s__tu',
        'at921s__tu',
        'rl900b__tu',
        'rl900c__tu',
        'rl933s__tu',
        'rl933b__tu',
        'rl933c__tu',
        'us900s__tu',
        'us900b__tu',
        'us900c__tu',
        'at999s__tu',
        'at999b__tu',
        'at921b__tu',
        'at900c__tu',
        'cvbk__tu',
        'archive__tu',
        'cvna__tu',
        'cvpl__tu',
        'cv_bc_prop__tu',
        'cv_cons_prop__tu',
        'cv_persloan_prop__tu',
        'syntheticfraud_score__tu',
        'vantagescore3__tu',
        'vantagescore4__tu', # score
        'credit_as_of_date__tu',
        'cv_bc_behav_seg__tu',
        'tu_creditbureaudate__tu',
        'at900b__tu',
        'linkchit__tu',
        'scr_alt_a_2__tu',
        'scr_link_a_2__tu',
        'scr_alt_b_2__tu',
        'scr_link_b_2__tu',
        'zipcodefiller__tu',
        'score_cvauto__tu',
        'score_newaccount__tu',
        'score_epd__tu',
        'at900s__tu',
        'ad13e__tu',
        'ad13d__tu',
        'ad13c__tu',
        'dateofbirth__tu',
        'firstname__tu',
        'middlename__tu',
        'lastname__tu',
        'address1__tu',
        'city__tu',
        'strname__tu',
        'state__tu',
        'zip5__tu',
        'zip4__tu',
        'ssn__tu',
        'dtmstampcreation__tu',
        'bkc225__tu',
        'dtmapproved__tu',
        'dtmdeclined__tu',
        'observationdate__tu',
        'tusequencenumber__tu',
        'analyticsmatchkey__tu',
        'segment__tu',
        'indflag__tu',
        'ad01a__tu',
        'ad01b__tu',
        'ad01c__tu',
        'creditasofdate__tu',
        'bkc224__tu',
        'ad01e__tu',
        'linkc014__tu',
        'bkc231__tu',
        'bkc232__tu',
        'bkc233__tu',
        'bkc234__tu',
        'bkc235__tu',
        'bkc252__tu',
        'bkc253__tu',
        'bkc254__tu',
        'linkb006__tu',
        'linkb007__tu',
        'linkc015__tu',
        'bkc223__tu',
        'linkc016__tu',
        'linkc022__tu',
        'linkc023__tu',
        'bkc255__tu',
        'bkc201__tu',
        'bkc202__tu',
        'bkc203__tu',
        'bkc204__tu',
        'bkc205__tu',
        'bkc222__tu',
        'ad01d__tu',
        'ad01f__tu',
        'ad13b__tu',
        'ad07a__tu',
        'ad05c__tu',
        'ad05d__tu',
        'ad05e__tu',
        'ad05f__tu',
        'ad05g__tu',
        'ad05h__tu',
        'ad06a__tu',
        'ad06b__tu',
        'ad06c__tu',
        'ad06d__tu',
        'ad08a__tu',
        'ad05a__tu',
        'ad09a__tu',
        'ad12a__tu',
        'ad12b__tu',
        'ad12c__tu',
        'ad12d__tu',
        'ad12e__tu',
        'ad12f__tu',
        'ad12g__tu',
        'ad12h__tu',
        'ad13a__tu',
        'ad05b__tu',
        'ad04h__tu',
        'ad01g__tu',
        'ad03b__tu',
        'ad01h__tu',
        'ad02a__tu',
        'ad02b__tu',
        'ad02c__tu',
        'ad02d__tu',
        'ad02e__tu',
        'ad02f__tu',
        'ad02g__tu',
        'ad02h__tu',
        'ad03a__tu',
        'ad03c__tu',
        'ad04g__tu',
        'ad03d__tu',
        'ad03e__tu',
        'ad03g__tu',
        'ad03h__tu',
        'ad04a__tu',
        'ad04b__tu',
        'ad04c__tu',
        'ad04d__tu',
        'ad04e__tu',
        'ad04f__tu',
        'au921s__tu',
        'ad03f__tu',
        'au921c__tu',
        'mt922b__tu',
        'he934d__tu',
        'he934s__tu',
        'he935b__tu',
        'he935c__tu',
        'he935d__tu',
        'he935s__tu',
        'mt920b__tu',
        'mt920c__tu',
        'mt920d__tu',
        'mt920s__tu',
        'mt922c__tu',
        'he934b__tu',
        'mt922d__tu',
        'mt922s__tu',
        'mt934b__tu',
        'mt934c__tu',
        'mt934d__tu',
        'mt934s__tu',
        'mt935b__tu',
        'mt935c__tu',
        'mt935d__tu',
        'mt935s__tu',
        'he934c__tu',
        'he922s__tu',
        'rl920c__tu',
        'cc100s__tu',
        'bc922d__tu',
        'bc922s__tu',
        'bc934b__tu',
        'bc934c__tu',
        'bc934d__tu',
        'bc934s__tu',
        'bc935b__tu',
        'bc935c__tu',
        'bc935d__tu',
        'bc935s__tu',
        'cc101b__tu',
        'he922d__tu',
        'cc102s__tu',
        'cc103s__tu',
        'cc104s__tu',
        'cc105s__tu',
        'he920b__tu',
        'he920c__tu',
        'he920d__tu',
        'he920s__tu',
        'he922b__tu',
        'he922c__tu',
        'rl920b__tu',
        'rl920d__tu',
        'bc922b__tu',
        'us934b__tu',
        'rt935d__tu',
        'rt935s__tu',
        'us920b__tu',
        'us920c__tu',
        'us920d__tu',
        'us920s__tu',
        'us922b__tu',
        'us922c__tu',
        'us922d__tu',
        'us922s__tu',
        'finscore__tu',
        'rt935b__tu',
        'score_cvpropensity__tu',
        'score_bankcard__tu',
        'us935s__tu',
        'us935d__tu',
        'us935c__tu',
        'us935b__tu',
        'us934s__tu',
        'us934d__tu',
        'us934c__tu',
        'au933s__tu',
        'rt935c__tu',
        'rt934s__tu',
        'rl920s__tu',
        'rl935d__tu',
        'rl922b__tu',
        'rl922c__tu',
        'rl922d__tu',
        'rl922s__tu',
        'rl934b__tu',
        'rl934c__tu',
        'rl934d__tu',
        'rl934s__tu',
        'rl935b__tu',
        'rl935c__tu',
        'rl935s__tu',
        'rt934d__tu',
        'rt920b__tu',
        'rt920c__tu',
        'rt920d__tu',
        'rt920s__tu',
        'rt922b__tu',
        'rt922c__tu',
        'rt922d__tu',
        'rt922s__tu',
        'rt934b__tu',
        'rt934c__tu',
        'bc922c__tu',
        'cc101s__tu',
        'bc920s__tu',
        'us921b__tu',
        'mt921b__tu',
        'mt921c__tu',
        'mt933s__tu',
        'mt933b__tu',
        'mt933c__tu',
        'mt999s__tu',
        'mt999b__tu',
        'rl921s__tu',
        'rl921b__tu',
        'rl921c__tu',
        'rl999s__tu',
        'bc920d__tu',
        'rt900s__tu',
        'rt900b__tu',
        'rt900c__tu',
        'rt921s__tu',
        'rt921b__tu',
        'rt921c__tu',
        'rt933s__tu',
        'rt933b__tu',
        'rt933c__tu',
        'rt999s__tu',
        'rt999b__tu',
        'mt921s__tu',
        'mt900c__tu',
        'mt900b__tu',
        'bc999b__tu',
        'au933b__tu',
        'au933c__tu',
        'au999s__tu',
        'au999b__tu',
        'bc921s__tu',
        'bc921b__tu',
        'bc921c__tu',
        'bc933s__tu',
        'bc933b__tu',
        'bc933c__tu',
        'he900s__tu',
        'mt900s__tu',
        'he900b__tu',
        'he900c__tu',
        'he921s__tu',
        'he921b__tu',
        'he921c__tu',
        'he933s__tu',
        'he933b__tu',
        'he933c__tu',
        'he999s__tu',
        'he999b__tu',
        'us921s__tu',
        'rl999b__tu',
        'us921c__tu',
        'au922c__tu',
        'at935d__tu',
        'at935e__tu',
        'at935f__tu',
        'at935s__tu',
        'at999c__tu',
        'au920b__tu',
        'au920c__tu',
        'au920d__tu',
        'bc920b__tu',
        'au935s__tu',
        'au935d__tu',
        'au935c__tu',
        'us933s__tu',
        'au935b__tu',
        'au934s__tu',
        'au934d__tu',
        'au934c__tu',
        'au934b__tu',
        'au922s__tu',
        'au922d__tu',
        'au920s__tu',
        'at935c__tu',
        'at935b__tu',
        'at934s__tu',
        'at922b__tu',
        'us933b__tu',
        'us933c__tu',
        'us999s__tu',
        'us999b__tu',
        'at920b__tu',
        'at920c__tu',
        'at920d__tu',
        'at920e__tu',
        'at920f__tu',
        'at934f__tu',
        'at920s__tu',
        'at922c__tu',
        'at922d__tu',
        'at922e__tu',
        'bc920c__tu',
        'at922f__tu',
        'at922s__tu',
        'at934b__tu',
        'at934c__tu',
        'at934d__tu',
        'at934e__tu',
        'au922b__tu',
        'linkd004__tu',
        'linkf011__tu',
        'linkf190__tu',
        'linkf012__tu',
        'linkd012__tu',
        'linkd013__tu',
        'linkf080__tu',
        'linkf013__tu',
        'linkf027__tu',
        'linkf186__tu',
        'linkf008__tu',
        'linkf182__tu',
        'linkf192__tu',
        'linkf009__tu',
        'linkf028__tu',
        'linkd011__tu',
        'linkf010__tu',
        'linkf029__tu',
        'linkc027__tu',
        'linkc031__tu',
        'linkc024__tu',
        'linkc029__tu',
        'linkc028__tu',
        'linkc033__tu',
        'linkc026__tu',
        'linkc048__tu',
        'linkc007__tu',
        'linkc020__tu',
        'linkc006__tu',
        'linkc032__tu',
        'linkc025__tu',
        'linkc034__tu',
        'linkc004__tu',
        'linkc043__tu',
        'linkc044__tu',
        'linkc045__tu',
        'linkc046__tu',
        'linkc047__tu',
        'linkc049__tu',
        'linkc005__tu',
        'linkc012__tu',
        'linkc011__tu',
        'linkc035__tu',
        'linkc050__tu',
        'linkc051__tu',
        'linkc009__tu',
        'linkc030__tu',
        'linkc038__tu',
        'linkc039__tu',
        'linkc040__tu',
        'linkc041__tu',
        'linkc042__tu',
        'linkc010__tu',
        'linkc037__tu',
        'linkc036__tu',
        'linkc008__tu',
        'linkf024__tu',
        'linkf049__tu',
        'linkf191__tu',
        'linkf025__tu',
        'linkf026__tu',
        'linkd001__tu',
        'linkb001__tu',
        'linkb013__tu',
        'linkb018__tu',
        'linkb016__tu',
        'linkb015__tu',
        'linkb014__tu',
        'linkb003__tu',
        'linkb004__tu',
        'linkb002__tu',
        'linkb005__tu',
        'linkd008__tu',
        'linkf081__tu',
        'linkf188__tu',
        'linkd010__tu',
        'linkf123__tu',
        'linkf124__tu',
        'linkb017__tu',
        'linkf184__tu',
        'linkf014__tu',
        'linkf016__tu',
        'linkf017__tu',
        'linkf213__tu',
        'linkf034__tu',
        'linkf135__tu',
        'linkf187__tu',
        'linkd014__tu',
        'linkd005__tu',
        'linkd009__tu',
        'linkd006__tu',
        'linkd007__tu',
        'linkd002__tu',
        'linkd003__tu',
        'linkf136__tu',
        'linkf030__tu',
        'linkb009__tu',
        'linkb008__tu',
        'linkb011__tu',
        'linkb010__tu',
        'linkf033__tu',
        # 2023-11-06
        'confirmationinputdob__ln',
        'educationinstitutionprivate__ln',
        'vantagescore4__tu', # lots of missing
        # 2023-11-07
        'intscore__ln',
    ]
    
    # load in list
    print('Loading list of starting features...')
    str_filename = 'df_cols_final.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/03_get_list_of_cols/{str_filename}'
    list_cols = list(pd.read_csv(str_uri)['feature'])
    # rm cols
    list_cols = [col for col in list_cols if col not in list_cols_ignore]
    
    # rm LN features
    print(f'Removing LN features...')
    list_cols = [col for col in list_cols if '__ln' not in col.lower()]
    
    # load list of LN
    print('Loading list of available LN features...')
    str_filename = 'df_ln.csv'
    str_uri = 's3://20231010-gen-xii/ad_hoc/05_check_features/df_ln.csv'
    list_cols_ln = list(pd.read_csv(str_uri)['feature'])
    
    # combine
    print('Adding LN features to list_cols...')
    list_cols = list_cols + list_cols_ln
    list_cols = list(dict.fromkeys(list_cols))
    
    # save
    print('Write to s3...')
    df = pd.DataFrame({'feature': list_cols})
    str_filename = 'df_list_features.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/01_lambda_get_starting_feats/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-starting-feats-1

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  103.4kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
3.8: Pulling from lambda/python
ca99af162351: Pulling fs layer
bc87a5a20b08: Pulling fs layer
432e3f1b14e4: Pulling fs layer
ca57dfa5425b: Pulling fs layer
5039b4c5136c: Pulling fs layer
d57015a4f27f: Pulling fs layer
ca57dfa5425b: Waiting
5039b4c5136c: Waiting
432e3f1b14e4: Verifying Checksum
432e3f1b14e4: Download complete
bc87a5a20b08: Verifying Checksum
bc87a5a20b08: Download complete
ca57dfa5425b: Verifying Checksum
ca57dfa5425b: Download complete
d57015a4f27f: Verifying Checksum
d57015a4f27f: Download complete
5039b4c5136c: Verifying Checksum
5039b4c5136c: Download complete
ca99af162351: Download complete
ca99af162351: Pull complete
bc87a5a20b08: Pull complete
432e3f1b14e4: Pull complete
ca57dfa5425b: Pull complete
5039b4c5136c: Pull complete
d57015a4f27f: Pull complete
Digest: sha256:ff1a2ff1d512a26197d556331eb985adeddd1287f4a930953dee985ca5d960e2
Status: Downloaded newer image for p

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 27.4 MB/s eta 0:00:00
Removing intermediate container ae43bde939c6
 ---> 15598a5f869f
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 8a3b778485bc
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 02d9948c642a
Removing intermediate container 02d9948c642a
 ---> b4b4b0cbd692
Successfully built b4b4b0cbd692
Successfully tagged genxii-ad-starting-feats-1:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-starting-feats-1' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-starting-feats-1]
d6e760bc9e9a: Preparing
6cc93fffdb41: Preparing
9ef8cce21df5: Preparing
99006831cc42: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
4fe51bf0bf5c: Waiting
fbbd8c1e2ec1: Waiting
97a787951169: Waiting
e703f2e518cc: Waiting
fe2359fe88f2: Waiting
e92756f7b561: Layer already exists
4fe51bf0bf5c: Layer already exists
fbbd8c1e2ec1: Layer already exists
fe2359fe88f2: Layer already exists
e703f2e518cc: Layer already exists
97a787951169: Layer already exists
9ef8cce21df5: Pushed
d6e760bc9e9a: Pushed
99006831cc42: Pushed
6cc93fffdb41: Pushed
latest: digest: sha256:929383cd4641117bc20e8a605ef810aa89fb7997dcbe0acf8cf28df476413cc6 size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 06 May 2024 15:54:12 GMT',
                                      'x-amzn-requestid': 'f1641486-41cc-485d-b163-32d5fba16d8f'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'f1641486-41cc-485d-b163-32d5fba16d8f',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-starting-feats-1:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '929383cd4641117bc20e8a605ef810aa89fb7997dcbe0acf8cf28df476413cc6',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-starting-feats',
 'FunctionName': 'genxii-ad-starting-feats',
 'LastModified': '2024-05-06T15:54:12.490+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-starting-feats'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1205',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 06 May 2024 15:54:13 GMT',
                                      'x-amzn-requestid': '8a59ae1d-e0eb-457c-8541-922870ff46b4'},
                      'HTTPStatusCode': 201,
                      'RequestId': '8a59ae1d-e

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)